# Chapter 3 — Computing Sensitivity in Practice
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local to Global Sensitivity*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Finite-difference SI computation, the U-curve for step-size selection,
correlated POIs, and the black-box sensitivity Jacobian template.

**Key claims tested:**
- Centered differences have O(h²) error; forward differences have O(h)
- U-curve minimum at h* ≈ (ε_mach)^(1/3) |p*|
- ODE solver tolerance dominates when h < solver_tol
- Correlated k,β in SIR: SI_k = SI_β (structural redundancy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings; warnings.filterwarnings('ignore')

FULL = False
print('Chapter 3: Computing Sensitivity in Practice')

In [ ]:
# ── Sensitivity Jacobian (Python port of sensitivity_jacobian.m) ───────────────
def sensitivity_jacobian(model, p_nom, n_q, h_scale=6e-6, h_min=1e-10):
    """
    Compute the normalized sensitivity Jacobian via centered finite differences.

    Parameters
    ----------
    model   : callable, p_vec -> array of shape (n_q,)
    p_nom   : array-like, nominal parameter vector (m,)
    n_q     : int, number of QOIs
    h_scale : float, step size scale factor (default 6e-6)
    h_min   : float, minimum step size (default 1e-10)

    Returns
    -------
    S : ndarray, shape (n_q, m) — normalized sensitivity indices
    """
    p_nom = np.asarray(p_nom, dtype=float)
    m     = len(p_nom)
    q0    = np.atleast_1d(model(p_nom)).astype(float)
    S     = np.zeros((n_q, m))

    for i in range(m):
        h     = max(h_scale * abs(p_nom[i]), h_min)
        p_p   = p_nom.copy(); p_p[i] += h
        p_m   = p_nom.copy(); p_m[i] -= h
        dq_dp = (np.atleast_1d(model(p_p)) - np.atleast_1d(model(p_m))) / (2*h)
        # Normalize: SI = (p/q) * dq/dp; use regularized form near zero
        for j in range(n_q):
            denom = max(abs(q0[j]), 1.0) if abs(q0[j]) < 0.01 else q0[j]
            S[j, i] = (p_nom[i] / denom) * dq_dp[j]
    return S

print('sensitivity_jacobian loaded.')

In [ ]:
# ── Verification Suite ─────────────────────────────────────────────────────────
print('=' * 60); print('VERIFICATION SUITE'); print('=' * 60)

# Test 1: SI for R0 = c*beta*tau_R, all = 1
p_nom = np.array([5.0, 0.06, 7.0])  # c, beta, tau_R
R0_fn = lambda p: np.array([p[0]*p[1]*p[2]])
S = sensitivity_jacobian(R0_fn, p_nom, 1)
for i, (name, expected) in enumerate(zip(['c','beta','tau_R'], [1,1,1])):
    err = abs(S[0,i] - expected)
    assert err < 10**-8, f'FAIL: SI_{name}={S[0,i]:.8f}'  # integer form: a test tolerance, not the RelTol the text recommends
    print(f'Test PASS: SI_{name}^R0 = {S[0,i]:.8f}  (exact={expected})')

# Test 2: Forward vs centered error order
def test_func(p): return np.array([np.sin(p[0]**2)])
p0 = np.array([1.3])
exact_d = 2*1.3*np.cos(1.3**2)
h_vals = np.logspace(-2, -8, 20)
err_cen = []; err_fwd = []
for h in h_vals:
    cen = (test_func([1.3+h])[0] - test_func([1.3-h])[0]) / (2*h)
    fwd = (test_func([1.3+h])[0] - test_func([1.3])[0]) / h
    err_cen.append(abs(cen - exact_d))
    err_fwd.append(abs(fwd - exact_d))
slope_cen = np.polyfit(np.log10(h_vals[2:10]), np.log10(err_cen[2:10]), 1)[0]
slope_fwd = np.polyfit(np.log10(h_vals[2:10]), np.log10(err_fwd[2:10]), 1)[0]
assert abs(slope_cen - 2) < 3/10, f'FAIL: centered slope={slope_cen:.2f}'
assert abs(slope_fwd - 1) < 3/10, f'FAIL: forward slope={slope_fwd:.2f}'
print(f'Test PASS: centered O(h^{slope_cen:.2f}), forward O(h^{slope_fwd:.2f})')

# Test 3: Correlated POIs — SI_c = SI_beta for SIR R0
SI_c, SI_beta = S[0,0], S[0,1]
assert abs(SI_c - SI_beta) < 10**-10, 'FAIL: structural redundancy'  # integer form: not the template's h_min
print(f'Test PASS: SI_c = SI_beta = {SI_c:.6f} (structural redundancy, c*beta)')

print(f'\nAll 5 tests PASSED.'); print('=' * 60)

In [ ]:
# ── Figure 1: U-curve for step-size selection ─────────────────────────────────
# q(p) = sin(p²) at p* = 1.3  (§3.3 worked example)
p0 = 1.3
exact_d = 2*p0*np.cos(p0**2)
h_vals = np.logspace(-1, -14, 200)
err_cen_plt = [abs((np.sin((p0+h)**2) - np.sin((p0-h)**2))/(2*h) - exact_d)
               for h in h_vals]
err_fwd_plt = [abs((np.sin((p0+h)**2) - np.sin(p0**2))/h - exact_d)
               for h in h_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_vals, err_cen_plt, 'steelblue', lw=2, label='Centered difference')
ax.loglog(h_vals, err_fwd_plt, 'coral',     lw=2, label='Forward difference')
ax.axvline(6e-6*p0, color='steelblue', ls='--', lw=1.2,
           label=fr'$h^* \approx 6\times10^{{-6}}|p^*|$')
ax.axhline(1e-13, color='gray', ls=':', lw=1, alpha=0.5)
ax.set_xlabel(r'Step size $h$', fontsize=12)
ax.set_ylabel(r'|Error in derivative|', fontsize=12)
ax.set_title(fr'U-curve: $q(p)=\sin(p^2)$ at $p^*={p0}$  (Figure 3.2)', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(1e-14, 1e0)
plt.tight_layout()
plt.savefig('ch03_fig1_ucurve.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch03_fig1_ucurve.png', dpi=150, bbox_inches='tight')
plt.show(); print('U-curve saved.')

In [ ]:
# ── The complex step (Section 3.3, sec:complexstep) ─────────────────────
# q'(p) ~ Im q(p + ih)/h: no subtraction, so no cancellation, and h can be 1e-20.
# Every number the subsection, Table 3.2 (tab:complexstep), the cost-table row and
# the new part of the Figure 3.1 caption print is recomputed and checked here.
# Self-contained: it does not rely on the cells above.
import math
import numpy as np
from decimal import Decimal, getcontext
from scipy.integrate import solve_ivp

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

def agrees_sig(x, shown, sig):
    """True when x rounds to the value the book prints, at its significant digits."""
    return abs(x - shown) <= 0.5 * 10.0 ** (math.floor(math.log10(abs(shown))) - sig + 1) * (1 + 1e-9)

def order_is(x, power):
    """True when x is within one decade of the printed order of magnitude `power`."""
    return x > 0 and abs(math.log10(x / power)) <= 1

# The closed-SIR peak prevalence of Figure 3.1, real or complex beta.
C_CS, TAUR_CS, N_CS, BETA_CS = 5.0, 7.0, 1000.0, 0.06
def i_max_cs(beta):
    r = C_CS * beta * TAUR_CS
    return N_CS * (1 - (1 + np.log(r)) / r)       # np.log accepts a complex argument

# Reference: the exact derivative in 40-digit decimal arithmetic at the double nearest 0.06,
# as in generate_fig_ucurve.py.
getcontext().prec = 40
_rD = Decimal(C_CS) * Decimal(BETA_CS) * Decimal(TAUR_CS)
D_REF = Decimal(N_CS) * _rD.ln() / (_rD * _rD) * Decimal(C_CS) * Decimal(TAUR_CS)
def rel_cs(x):
    return float(abs(Decimal(float(x)) - D_REF) / D_REF)

cen = lambda h: (i_max_cs(BETA_CS + h) - i_max_cs(BETA_CS - h)) / (2 * h)
cstep = lambda h: i_max_cs(BETA_CS + 1j * h).imag / h
EPS_CS = float(np.finfo(float).eps)
H_STAR_CS = EPS_CS ** (1 / 3) * BETA_CS

print(' h          centered     complex step')
for h in (1e-4, H_STAR_CS, 1e-8, 1e-12, 1e-20):
    print(f' {h:9.2e}  {rel_cs(cen(h)):10.2e}  {rel_cs(cstep(h)):10.2e}')

# Table 3.2, row by row.
assert agrees_sig(rel_cs(cen(1e-4)), 3.4e-7, 2) and agrees_sig(rel_cs(cstep(1e-4)), 3.4e-7, 2), 'FAIL: row h = 1e-4'  # book: ch03_body.tex "$10^{-4}$                 & $3.4\times10^{-7}$ & $3.4\times10^{-7}$"
assert agrees_sig(H_STAR_CS, 3.63e-7, 3) and order_is(rel_cs(cen(H_STAR_CS)), 1e-11) and agrees_sig(rel_cs(cstep(H_STAR_CS)), 4.5e-12, 2), 'FAIL: row h*'  # book: ch03_body.tex "$h^* = 3.63\times10^{-7}$ & $\sim 10^{-11}$    & $4.5\times10^{-12}$"
assert order_is(rel_cs(cen(1e-12)), 1e-6) and order_is(rel_cs(cstep(1e-12)), 1e-16), 'FAIL: row h = 1e-12'  # book: ch03_body.tex "$10^{-12}$                & $\sim 10^{-6}$     & $\sim 10^{-16}$"
H_TINY = 1e-20                                       # the complex step's usual step
assert BETA_CS + H_TINY == BETA_CS and BETA_CS - H_TINY == BETA_CS and cen(H_TINY) == 0, 'FAIL: beta* +- h'
assert rel_cs(cen(1e-20)) == 1.0 and order_is(rel_cs(cstep(1e-20)), 1e-16), 'FAIL: row h = 1e-20'  # book: ch03_body.tex "$10^{-20}$                & $1$                & $\sim 10^{-16}$"
assert agrees(BETA_CS, 0.06, 2), 'FAIL: beta* in the caption'  # book: ch03_body.tex "$\beta^* = 0.06$ for the centered difference~\eqref{eq:FD2} and"
assert BETA_CS + 1e-20 == BETA_CS and rel_cs(cen(1e-20)) == 1.0, 'FAIL: caption, h = 1e-20'  # book: ch03_body.tex "reproducible. At $h = 10^{-20}$ both $\beta^* + h$ and"

# The text after the table.
assert abs(rel_cs(cen(1e-4)) - rel_cs(cstep(1e-4))) / rel_cs(cstep(1e-4)) < 0.01, 'FAIL: equal truncation'  # book: ch03_body.tex "At $h = 10^{-4}$ the two errors agree"
_hh = np.logspace(-20.0, -3.0, 341)
_ecs = np.array([rel_cs(cstep(s)) for s in _hh])
_floor = _ecs[_hh <= 1e-9].max()
assert _floor < 1e-15 and rel_cs(cstep(1e-8)) < 1e-14 and rel_cs(cstep(1e-7)) > 1e-14, 'FAIL: rounding reached near 1e-8'  # book: ch03_body.tex "as $h^2$ until it meets rounding near $h = 10^{-8}$, and then it stays"
assert _ecs[0] < 1e-15 and _hh[0] == 1e-20, 'FAIL: flat to 1e-20'  # book: ch03_body.tex "flat all the way to $h = 10^{-20}$ (Figure~\ref{fig:ucurve})"
_ratio = rel_cs(cen(1e-12)) / rel_cs(cen(H_STAR_CS))
assert 4 <= math.log10(rel_cs(cen(1e-12)) / rel_cs(cen(H_STAR_CS))) <= 6 and cen(1e-20) == 0.0, f'FAIL: five orders ({_ratio:.1e})'  # book: ch03_body.tex "$h = 10^{-12}$, and fails completely at $h = 10^{-20}$. At that last step"
dI_cs = cstep(1e-20)
SI_cs = BETA_CS / i_max_cs(BETA_CS + 1e-20j).real * dI_cs
_SI_ref = float(Decimal(BETA_CS) * D_REF / Decimal(i_max_cs(BETA_CS)))
assert agrees(dI_cs, 5888.391625, 6), f'FAIL: dI/dbeta = {dI_cs:.6f}'  # book: ch03_body.tex "the complex step gives $\partial I_{\max}/\partial\beta = 5888.391625$"
assert agrees(SI_cs, 2.072088, 6) and agrees(_SI_ref, 2.072088, 6), f'FAIL: SI = {SI_cs:.7f}'  # book: ch03_body.tex "and $\SI{\beta}{I_{\max}} = 2.072088$, both correct in every digit shown."
assert agrees(cen(H_STAR_CS), 5888.391625, 6) and agrees(BETA_CS / i_max_cs(BETA_CS) * cen(H_STAR_CS), 2.072088, 6), 'FAIL: centered at h*'  # book: ch03_body.tex "the complex step gives $\partial I_{\max}/\partial\beta = 5888.391625$"
assert i_max_cs(BETA_CS + 1e-20j).real == i_max_cs(BETA_CS), 'FAIL: real part is the baseline'

# The subsection's opening paragraphs and the Figure 3.1 caption.
assert BETA_CS + 1e-20 == BETA_CS, 'FAIL: p* + h rounds to p*'  # book: ch03_body.tex "truncation term drops under rounding. A step of $h = 10^{-20}$ is a"
assert order_is(_floor, 1e-16) and rel_cs(cstep(1e-8)) < 1e-14 and _ecs[0] < 1e-15, 'FAIL: caption'  # book: ch03_body.tex "$10^{-16}$, near $h = 10^{-8}$ and stays flat as $h$ shrinks to"
assert _hh[0] == 1e-20, 'FAIL: caption range'  # book: ch03_body.tex "$10^{-20}$. Errors are measured against the closed form evaluated in"
assert getcontext().prec == 40, 'FAIL: 40 digits'

# The worked case's parameters (the text restates the Figure 3.1 values).
assert agrees(C_CS, 5.0, 0) and agrees(TAUR_CS, 7.0, 0) and agrees(BETA_CS, 0.06, 2), 'FAIL: worked-case values'  # book: ch03_body.tex "of Figure~\ref{fig:ucurve}, with $c = 5$, $\tau_R = 7$, $\beta^* = 0.06$"
assert N_CS == 1000, 'FAIL: N'  # book: ch03_body.tex "and $N = 1000$. The code needs one change: pass $\beta^* + ih$ in place"

# The Laboratory's ODE model: SIR with demography, I(60), RelTol = 1e-10.
def sir_I60_cs(beta, rtol=1e-10, atol=1e-12, c=5.0, tau_R=7.0, tau_m=1e4, N=1000.0, T=60.0):
    nu_m = 1.0 / tau_m
    def rhs(t, y):
        S, I, R = y
        alpha = c * beta * S / N
        return [nu_m * N - alpha * I - nu_m * S, alpha * I - I / tau_R - nu_m * I, I / tau_R - nu_m * R]
    y0 = np.array([999.0, 1.0, 0.0], dtype=complex if isinstance(beta, complex) else float)
    sol = solve_ivp(rhs, [0.0, T], y0, rtol=rtol, atol=atol, dense_output=True)
    return sol.sol(T)[1]

_hcs = 1e-20
_Ic = sir_I60_cs(BETA_CS + 1j * _hcs)
_I0 = sir_I60_cs(BETA_CS)
SI60_cs = BETA_CS / _I0 * _Ic.imag / _hcs
_hc = 6e-6 * BETA_CS                                  # the template's centered step
SI60_cd = BETA_CS / _I0 * (sir_I60_cs(BETA_CS + _hc) - sir_I60_cs(BETA_CS - _hc)) / (2 * _hc)
_gap = abs(SI60_cs - SI60_cd) / abs(SI60_cs)
print(f'ODE model: complex-step and centered SI_beta^I(60) differ by {_gap:.1e} (relative)')
assert agrees(sir_I60_cs.__defaults__[2], 5.0, 0), 'FAIL: lab c'  # book: ch03_body.tex "model with demography of Section~\ref{sec:ch3matlab}, with $c = 5$,"
assert agrees(BETA_CS, 0.06, 2) and agrees(sir_I60_cs.__defaults__[3], 7.0, 0) and sir_I60_cs.__defaults__[4] == 1e4 and sir_I60_cs.__defaults__[5] == 1000, 'FAIL: lab beta, tau_R, tau_m, N'  # book: ch03_body.tex "$\beta = 0.06$, $\tau_R = 7$, $\tau_m = 10^4$ and $N = 1000$. Give the"
assert sir_I60_cs.__defaults__[0] == 1e-10, 'FAIL: RelTol'  # book: ch03_body.tex "$\texttt{RelTol} = 10^{-10}$, the complex-step value of"
assert sir_I60_cs.__defaults__[1] == 1e-12, 'FAIL: AbsTol'  # book: ch03_body.tex "absolute tolerance $10^{-12}$) integrates"
assert _hc == 6e-6 * BETA_CS, 'FAIL: relative centered step'  # book: ch03_body.tex "at the relative step $6\times10^{-6}$ to about"
assert _hcs == 1e-20 and sir_I60_cs.__defaults__[6] == 60, 'FAIL: h, T = 60'  # book: ch03_body.tex "$\SI{\beta}{I(60)}$ at $h = 10^{-20}$ agrees with the centered difference"
GAP_LO, GAP_HI = 10.0 ** -11, 10.0 ** -8               # "about nine significant digits"
assert GAP_LO < _gap < GAP_HI, f'FAIL: about nine digits ({_gap:.1e})'
# The solver tolerance, not h, limits the accuracy: loosen RelTol and the complex step moves with it.
RT_TIGHT, RT_LOOSE, ATOL_FAC = 10.0 ** -12, 10.0 ** -6, 10.0 ** -2
_SIc = lambda rt: BETA_CS / sir_I60_cs(BETA_CS, rt, rt * ATOL_FAC) * sir_I60_cs(BETA_CS + 1j * _hcs, rt, rt * ATOL_FAC).imag / _hcs
_best = _SIc(RT_TIGHT)
assert abs(_SIc(RT_LOOSE) - _best) / abs(_best) > GAP_HI and abs(SI60_cs - _best) / abs(_best) < GAP_HI, 'FAIL: tolerance limits the accuracy'

# Two conditions: abs breaks the complex step silently.  q(p) = |p|^3 at p = -2, q'(-2) = 3p|p| = -12.
q_abs3 = lambda p: np.abs(p) ** 3
P_ABS = -2.0
_cs_bad = q_abs3(P_ABS + 1j * H_TINY).imag / H_TINY
def cabs(z):
    """Complex-safe absolute value: z when Re z >= 0, -z otherwise."""
    return z if np.real(z) >= 0 else -z
_cs_good = (cabs(P_ABS + 1j * H_TINY) ** 3).imag / H_TINY
assert P_ABS == -2.0 and _cs_bad == 0.0 and _cs_good == -12.0, 'FAIL: |p|^3'  # book: ch03_body.tex "itself. For $q(p) = |p|^3$ at $p = -2$, NumPy's \texttt{abs} returns the"
try:
    (1 + 1j) < 0
    _py_refuses = False
except TypeError:
    _py_refuses = True
assert _py_refuses and (np.complex128(1 + 5j) > np.complex128(1 + 1j)), 'FAIL: comparisons'

# Cost: a complex multiplication is four real multiplications and two additions.
_a, _b, _c, _d = 1.5, -0.25, 3.0, 0.75
assert complex(_a, _b) * complex(_c, _d) == complex(_a * _c - _b * _d, _a * _d + _b * _c), 'FAIL: complex product'
print(f'Complex step PASS: dI/dbeta = {dI_cs:.6f}, SI = {SI_cs:.6f}, floor {_floor:.1e}; '
      f'|p|^3 at -2: np.abs gives {_cs_bad}, safe abs gives {_cs_good}')

In [ ]:
# ── Figure 2: ODE solver tolerance effect on U-curve ──────────────────────────
# Shows why tightening RelTol matters (new §3.3 box)
def sir_I60(p_vec, rtol=1e-6):
    """Return I(60) for SIR model with given beta = p_vec[0]."""
    c, beta, tau_R = 5.0, p_vec[0], 7.0
    N = 1000
    def rhs(t, y):
        S, I, R = y
        dS = -c*beta*S*I/N; dI = c*beta*S*I/N - I/tau_R; dR = I/tau_R
        return [dS, dI, dR]
    sol = solve_ivp(rhs, [0, 60], [999,1,0], rtol=rtol, atol=rtol*1e-3,
                   dense_output=True)
    return np.array([sol.sol(60)[1]])

beta0 = np.array([0.06])
h_vals_ode = np.logspace(-3, -10, 60)

err_tight = []
err_loose = []
exact_I60 = sir_I60(beta0, rtol=1e-12)[0]
for h in h_vals_ode:
    try:
        cen_tight = (sir_I60(beta0*(1+h), rtol=1e-10)[0] - sir_I60(beta0*(1-h), rtol=1e-10)[0]) / (2*h*beta0[0])
        cen_loose = (sir_I60(beta0*(1+h), rtol=1e-4)[0]  - sir_I60(beta0*(1-h), rtol=1e-4)[0])  / (2*h*beta0[0])
        err_tight.append(abs(cen_tight*beta0[0]/exact_I60 - 1))
        err_loose.append(abs(cen_loose*beta0[0]/exact_I60 - 1))
    except:
        err_tight.append(np.nan); err_loose.append(np.nan)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_vals_ode, err_tight, 'steelblue', lw=2, label='Tight: RelTol=1e-10')
ax.loglog(h_vals_ode, err_loose, 'coral',     lw=2, label='Loose: RelTol=1e-4')
ax.set_xlabel(r'Finite-difference step size $h$', fontsize=12)
ax.set_ylabel('Relative error in SI', fontsize=12)
ax.set_title('ODE solver tolerance dominates below RelTol\n(new §3.3 box)', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.3)
ax.axvline(1e-4, color='coral', ls=':', lw=1.2, alpha=0.8,
           label='Loose tolerance floor')
plt.tight_layout()
plt.savefig('ch03_fig2_odetolerance.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch03_fig2_odetolerance.png', dpi=150, bbox_inches='tight')
plt.show(); print('ODE tolerance figure saved.')

In [ ]:
# ── Full Sensitivity Jacobian for SIR ─────────────────────────────────────────
T_LAB = 60  # days, the Laboratory's QOI time (named because the text states it)
def sir_qoi(p_vec):
    """SIR model with demography, eq. (sirS)-(sirR): return [S(60), I(60), R(60)] given p=[c,beta,tau_R,tau_m].
    Births nu_m N balance removals nu_m S, nu_m I, nu_m R, with nu_m = 1/tau_m; alpha = c beta S/N."""
    c, beta, tau_R, tau_m = p_vec
    N = 1000
    nu_m = 1.0 / tau_m
    def rhs(t, y):
        S, I, R = y
        alpha = c*beta*S/N                          # force from infection
        return [nu_m*N - alpha*I - nu_m*S, alpha*I - I/tau_R - nu_m*I, I/tau_R - nu_m*R]
    sol = solve_ivp(rhs, [0, T_LAB], [999,1,0], rtol=1e-10, atol=1e-12,
                   dense_output=True)
    return sol.sol(T_LAB)

p_sir = np.array([5.0, 0.06, 7.0, 10_000.0])  # c, beta, tau_R, tau_m (days)
S_jac = sensitivity_jacobian(sir_qoi, p_sir, 3)

qoi_names = ['S(60)', 'I(60)', 'R(60)']
poi_names = ['c', 'β', 'τ_R', 'τ_m']
print('Sensitivity Jacobian (rows=QOIs, cols=POIs):')
print(f'       {"  ".join(f"{n:>8s}" for n in poi_names)}')
for i, qname in enumerate(qoi_names):
    row = '  '.join(f'{S_jac[i,j]:8.4f}' for j in range(4))
    print(f'  {qname}: {row}')

print('\nNote: SI_c ≈ SI_β for I(60) (structural redundancy confirmed)')
assert abs(S_jac[1,0] - S_jac[1,1]) < 0.01, 'FAIL: c vs beta SI differ'
assert np.all(S_jac[:, 3] != 0) and S_jac[1, 3] < 0, 'FAIL: tau_m does not enter the lab model'

In [ ]:
# ── Verification Suite, part 2: the numbers Chapter 3 prints ──────────────────
# Each check recomputes a number the chapter prints.  A comment  # book: FILE "TEXT"  ties a
# check to the line of ch03_body.tex holding TEXT, so the check accounts for
# that number, and only that number, as reproduced.  Values the text states as
# givens are bound to names here, each with the value the text prints.
import numpy as np

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

def agrees_sig(x, shown, sig):
    """True when x rounds to the value the book prints, at its significant digits."""
    import math
    return abs(x - shown) <= 0.5 * 10.0 ** (math.floor(math.log10(abs(shown))) - sig + 1) * (1 + 1e-9)

EPS = float(np.finfo(float).eps)
assert agrees_sig(EPS, 2.2e-16, 2), 'FAIL: eps_mach'  # book: ch03_body.tex "\approx 2.2 \times 10^{-16}"
assert agrees_sig(np.sqrt(EPS), 1.5e-8, 2), 'FAIL: sqrt(eps)'  # book: ch03_body.tex "\approx 1.5 \times 10^{-8}\cdot|p^*|"
assert agrees_sig(EPS**(1/3), 6e-6, 1), 'FAIL: cbrt(eps)'  # book: ch03_body.tex "\approx 6 \times 10^{-6}\cdot" "then $6 \times 10^{-6}|p" "$h^* \approx 6\times10^{-6}|\beta^*|$"
P_STAR_RULE = 0.3                                   # the worked case after eq. (3.9)
assert agrees_sig(EPS**(1/3) * P_STAR_RULE, 1.8e-6, 2), 'FAIL: h for p* = 0.3'  # book: ch03_body.tex "$h \approx 1.8 \times 10^{-6}$"

# The curvature note of Chapter 3: q = sqrt(p) has index 1/2, and the second-order term has the
# opposite sign, so a 10% rise in p raises q by less than the first-order 5%.
P_RISE_SQRT = 1.10                                  # the 10% rise, as a factor
_pct_sqrt = (np.sqrt(P_RISE_SQRT) - 1) * 100
assert agrees(_pct_sqrt, 4.88, 2) and _pct_sqrt < 5, 'FAIL: sqrt(1.1) - 1'  # book: ch03_body.tex "raises $q$ by $4.88\%$, not $5\%$"

# Figure 3.1 (fig_ucurve): closed-SIR peak prevalence and its exact derivative
C3, TAUR3, N3, BETA3 = 5.0, 7.0, 1000.0, 0.06
def i_max(beta):
    r = C3 * beta * TAUR3
    return N3 * (1 - (1 + np.log(r)) / r)
_r = C3 * BETA3 * TAUR3
dImax = N3 * np.log(_r) / _r**2 * C3 * TAUR3
_s = 1e-5                                           # an exact-derivative cross-check
assert abs((i_max(BETA3 + _s) - i_max(BETA3 - _s)) / (2*_s) - dImax) / dImax < 10**-8, 'FAIL: closed form'  # integer form: a cross-check tolerance
assert agrees(dImax, 5888.391625, 6), f'FAIL: dImax/dbeta = {dImax:.6f}'  # book: ch03_body.tex "= 5888.391625$ is known"
h_star = EPS**(1/3) * abs(BETA3)
assert agrees_sig(h_star, 3.63e-7, 3), f'FAIL: h* = {h_star:.4e}'  # book: ch03_body.tex "3.63\times10^{-7}$, marked"
assert agrees_sig(h_star, 3.6e-7, 2), f'FAIL: h* = {h_star:.4e}'  # book: ch03_body.tex "step of $h \approx 3.6 \times 10^{-7}$" "$h^* \approx 3.6\times10^{-7}$. For the closed-form peak"
assert agrees_sig(3.6e-7 / 1e-12, 3.6e5, 2), 'FAIL: rounding ratio'  # book: ch03_body.tex "= 3.6\times10^5$ times larger"
# Self-check in Section 3.3, part (b): on the closed-form peak of Figure 3.1, the centered quotient at h = 1e-12 still
# has a relative error near 1e-6, and the forward and centered quotients differ by far less than 50%.
_rel_c = lambda s: abs((i_max(BETA3 + s) - i_max(BETA3 - s)) / (2*s) - dImax) / dImax
_gap_fc = lambda s: abs((i_max(BETA3 + s) - i_max(BETA3)) / s - (i_max(BETA3 + s) - i_max(BETA3 - s)) / (2*s)) / dImax
assert abs(np.log10(_rel_c(1e-12) / 1e-6)) <= 1 and _gap_fc(1e-12) < 0.5, f'FAIL: error at 1e-12 = {_rel_c(1e-12):.2e}'  # book: ch03_body.tex "so at $h = 10^{-12}$ it is" "$10^{-6}$, far below $50\%$"
# The measured optimum and its error are set by rounding noise, so only the robust
# claims are asserted: the measured optimum is within a factor 2 of h*, and its
# error is below 1e-11.  (The printed digits themselves are not checked.)
_hgrid = np.logspace(-14.0, -3.0, 221)
_err = np.array([abs((i_max(BETA3 + s) - i_max(BETA3 - s)) / (2*s) - dImax) / dImax for s in _hgrid])
_k = int(_err.argmin())
assert 1/2 < _hgrid[_k] / h_star < 2 and _err[_k] < 1e-11, f'FAIL: optimum {_hgrid[_k]:.3e}, error {_err[_k]:.3e}'

# Laboratory: why the step sweep does not start at h = 1e-1
assert agrees(BETA3 - 1e-1, -0.04, 2), 'FAIL: left stencil point'  # book: ch03_body.tex "$\beta - h = -0.04$"

# Self-check in Section 3.5: correlated POIs, omega_{c beta} = 0.4
C_CB = 0.4
assert agrees(1 + 1 * C_CB, 1.4, 1), 'FAIL: total SI'  # book: ch03_body.tex "1 + 1 \times 0.4 = 1.4" "approximately $1.4\%$"
# Self-check in Section 3.4: the exact relative change of R0 = c beta tau_R under joint relative changes is
# dc/c + db/b + (dc/c)(db/b); the product term against the first-order sum, for two 1% and two 20% increases.
_jc = lambda d: (C3 * (1 + d) * BETA3 * (1 + d) * TAUR3) / (C3 * BETA3 * TAUR3) - 1
assert agrees(100 * (_jc(0.01) - 2 * 0.01), 0.01, 2) and round(100 * 2 * 0.01) == 2 and round(100 * (_jc(0.2) - 2 * 0.2)) == 4 and round(100 * 2 * 0.2) == 40, 'FAIL: cross term'  # book: ch03_body.tex "small for small changes ($0.01\%$ against $2\%$"
# Self-check in Section 3.2: 20 POIs and 5 QOIs, 45 s and then 2 min per evaluation.
N_P_SC, N_Q_SC, T_EVAL_SC = 20, 5, 45
assert 2 * N_P_SC + 1 == 41 and (2 * N_P_SC + 1) * T_EVAL_SC == 1845 and round(1845 / 60) == 31, 'FAIL: cost'  # book: ch03_body.tex "$41 \times 45 = 1{,}845$~seconds"
assert max(n for n in range(1, 1000) if (2 * n + 1) * 2 <= 24 * 60) == 359, 'FAIL: 24-hour bound'
# Self-check in Section 3.6 (the template's step floor): h = h_scale * max(|p|, h_min/h_scale) at p = 0 is h_min.
_h_tmpl = lambda pv, h_scale, h_min: h_scale * max(abs(pv), h_min / h_scale)
assert 6e-6 * abs(0.0) == 0 and np.isclose(_h_tmpl(0.0, 6e-6, 1e-10), 1e-10, rtol=1e-12, atol=0), 'FAIL: step floor'  # book: ch03_body.tex "The floor $h_{\min} = 10^{-10}$ prevents this." "it is just the fixed small value $10^{-10}$."
# Section 3.3: the default tolerances.  SciPy's are read from the library: solve_ivp defaults to
# the RK45 method, whose rtol and atol defaults are the values the text prints.  (MATLAB's ode45
# has the same defaults; they cannot be read from Python.)
import inspect
from scipy.integrate import RK45
_rk45 = inspect.signature(RK45).parameters
assert inspect.signature(solve_ivp).parameters['method'].default == 'RK45', 'FAIL: default method'
assert (_rk45['rtol'].default, _rk45['atol'].default) == (1e-3, 1e-6), 'FAIL: default tolerances'  # book: ch03_body.tex "$\texttt{RelTol} = 10^{-3}$ and $\texttt{AbsTol} = 10^{-6}$"
print(f'Part 2 PASS: eps = {EPS:.2e}; dImax/dbeta = {dImax:.6f}; h* = {h_star:.3e}; '
      f'measured optimum {_hgrid[_k]:.3e} (error {_err[_k]:.2e})')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 3,
    'SI_R0': {'c': 1.0, 'beta': 1.0, 'tau_R': 1.0},
    'sensitivity_jacobian_SIR_I60': float(S_jac[1,1]),
    'fd_slope_centered': float(slope_cen),
    'fd_slope_forward': float(slope_fwd),
    'figures': ['ch03_fig1_ucurve.pdf', 'ch03_fig2_odetolerance.pdf']
}
print('Results:', RESULTS)

try:
    from google.colab import files
    for f in RESULTS['figures']: files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')

### Appendix F: the delta method with correlated inputs
Checks the numbers of Appendix F's paragraph *The delta method with correlated inputs*, its Table F.1 (sigma of R0 by the delta method and by a seeded Gaussian-copula Monte Carlo at corr(beta, tau_R) = -0.5, 0, 0.5) and Self-check F.3(b). Each check is tagged with the line of `appendix_f.tex` it reproduces.

In [ ]:
# ── Appendix F: the delta method with correlated inputs ──────────────────────────
# Recomputes every number of the paragraph "The delta method with correlated inputs", its
# Table F.1 (label tab:appF:deltacorr, generated by this cell), and Self-check F.3(b) in
# appendix_f.tex.  Each check names its line with
#     # book: appendix_f.tex "text on that line"
# Inputs (Section F.2): R0 = c beta tau_R, c = 5, beta mean 0.06 and sd 0.02, tau_R mean 7
# and sd 3.  Monte Carlo: uniform marginals with those moments, coupled by a Gaussian copula.
import numpy as np
from fractions import Fraction as Fr
from scipy.special import ndtr          # standard normal CDF

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

C_DC, MU_B, SD_B, MU_T, SD_T = 5.0, 0.06, 0.02, 7.0, 3.0
TOL_EXACT_DC, TOL_MC_DC = 1e-12, 5e-3    # two forms agree to rounding; MC against quadrature, relative
R0_NOM = C_DC * MU_B * MU_T
grad_r0 = np.array([C_DC * MU_T, C_DC * MU_B])          # (dR0/dbeta, dR0/dtau_R) at the means

def delta_var_r0(rho):
    """(grad)^T Sigma (grad), eq. (appF:deltacov), for corr(beta, tau_R) = rho."""
    sig = np.array([[SD_B**2, rho*SD_B*SD_T], [rho*SD_B*SD_T, SD_T**2]])
    return float(grad_r0 @ sig @ grad_r0)

def delta_cv2_r0(rho):
    """Relative form, eq. (appF:deltacv): both indices are 1 for the product."""
    cv_b, cv_t = SD_B / MU_B, SD_T / MU_T
    return cv_b**2 + cv_t**2 + 2 * rho * cv_b * cv_t

# Relative form: CV_beta = 1/3, CV_tau = 3/7, coefficients 1/9, 9/49, 2/7, exactly
cvb, cvt = Fr(2, 100) / Fr(6, 100), Fr(3, 7)
assert (cvb.numerator, cvb.denominator, cvt.numerator, cvt.denominator) == (1, 3, 3, 7)  # book: appendix_f.tex "both indices are $1$, $\CV_\beta = 1/3$ and $\CV_{\tau_R} = 3/7$, so"
assert (cvb**2, cvt**2, 2*cvb*cvt) == (Fr(1, 9), Fr(9, 49), Fr(2, 7))   # the three coefficients
assert agrees(R0_NOM**2, 4.41, 2)  # book: appendix_f.tex "Multiplying by $\cR_0^2 = 4.41$ gives"
v0, slope = delta_var_r0(0.0), delta_var_r0(1.0) - delta_var_r0(0.0)
assert agrees(v0, 1.30, 2) and agrees(slope, 1.26, 2) and agrees(v0, 1.30, 2)  # book: appendix_f.tex "$\Var(\cR_0) \approx 1.30 + 1.26\,\corr(\beta,\tau_R)$, which is the $1.30$"
for rho in (-0.5, 0.0, 0.3, 0.5):                        # the two forms agree
    assert abs(delta_cv2_r0(rho) * R0_NOM**2 - delta_var_r0(rho)) < TOL_EXACT_DC
assert agrees(delta_var_r0(-0.5), 0.67, 2)  # book: appendix_f.tex "The variance falls to $0.67$, and"
sd_delta = {rho: np.sqrt(delta_var_r0(rho)) for rho in (-0.5, 0.0, 0.5)}
assert agrees(sd_delta[0.0], 1.14, 2) and agrees(sd_delta[-0.5], 0.82, 2)  # book: appendix_f.tex "falls from $1.14$ to $0.82$."

# Uniform marginals with the stated moments: half-width sqrt(3) sigma, eq. (appF:uniformab)
hw_b, hw_t = np.sqrt(3) * SD_B, np.sqrt(3) * SD_T
assert agrees(MU_B - hw_b, 0.0254, 4) and agrees(MU_B + hw_b, 0.0946, 4)  # book: appendix_f.tex "$\beta \sim \mathcal{U}(0.0254, 0.0946)$ and"
assert agrees(MU_T - hw_t, 1.80, 2) and agrees(MU_T + hw_t, 12.20, 2)  # book: appendix_f.tex "$\tau_R \sim \mathcal{U}(1.80, 12.20)$. A Gaussian copula"
copula_r = lambda rho: 2 * np.sin(np.pi * rho / 6)
assert agrees(copula_r(-0.5), -0.518, 3)  # book: appendix_f.tex "it is $r \approx -0.518$."

N_DC = 10**6
assert N_DC == 1e6  # book: appendix_f.tex "with $10^6$ draws at each correlation."
def mc_r0(rho, seed=20261107):
    """Seeded Monte Carlo sample of R0 with corr(beta, tau_R) = rho via a Gaussian copula."""
    g = np.random.default_rng(seed)
    z1, z3 = g.standard_normal(N_DC), g.standard_normal(N_DC)
    r = copula_r(rho)
    z2 = r * z1 + np.sqrt(1 - r * r) * z3
    beta = MU_B - hw_b + 2 * hw_b * ndtr(z1)
    tau = MU_T - hw_t + 2 * hw_t * ndtr(z2)
    return beta, tau, C_DC * beta * tau

def quad_var_r0(rho, n=120):
    """The same variance by Gauss-Hermite quadrature in the two normals: a second route."""
    x, w = np.polynomial.hermite_e.hermegauss(n); w = w / w.sum()
    X, Z = np.meshgrid(x, x, indexing='ij'); W = np.outer(w, w); r = copula_r(rho)
    q = C_DC * (MU_B - hw_b + 2*hw_b*ndtr(X)) * (MU_T - hw_t + 2*hw_t*ndtr(r*X + np.sqrt(1-r*r)*Z))
    m = (W * q).sum()
    return float((W * q * q).sum() - m * m)

sd_mc = {}
for rho in (-0.5, 0.0, 0.5):
    b, t, q = mc_r0(rho)
    # the printed map r = 2 sin(pi corr / 6) gives the uniforms the target Pearson correlation
    assert copula_r(rho) == 2 * np.sin(np.pi * rho / 6) and abs(np.corrcoef(b, t)[0, 1] - rho) < TOL_MC_DC  # book: appendix_f.tex "range. The choice $r = 2\sin\bigl(\pi\corr(\beta,\tau_R)/6\bigr)$ gives the"
    sd_mc[rho] = float(q.std(ddof=1))
    assert abs(sd_mc[rho]**2 / quad_var_r0(rho) - 1) < TOL_MC_DC, 'FAIL: MC vs quadrature'
assert agrees(sd_delta[-0.5], 0.82, 2) and agrees(sd_mc[-0.5], 0.87, 2)  # book: appendix_f.tex "$-0.5$ & $0.82$ & $0.87$"
assert agrees(sd_delta[0.0], 1.14, 2) and agrees(sd_mc[0.0], 1.18, 2)  # book: appendix_f.tex "$\phantom{-}0.0$ & $1.14$ & $1.18$"
assert agrees(sd_delta[0.5], 1.39, 2) and agrees(sd_mc[0.5], 1.42, 2)  # book: appendix_f.tex "$\phantom{-}0.5$ & $1.39$ & $1.42$"

# Reading the table
RHO_LO_F, RHO_HI_F = -0.5, 0.5
shifts = [abs(d[r] - d[0.0]) for d in (sd_delta, sd_mc) for r in (-0.5, 0.5)]
gaps = [sd_mc[r] - sd_delta[r] for r in (-0.5, 0.0, 0.5)]
assert abs(RHO_LO_F) == RHO_HI_F == 0.5 and agrees(min(shifts), 0.24, 2) and agrees(max(shifts), 0.32, 2)  # book: appendix_f.tex "moves $\sigma_{\cR_0}$ by $0.24$ to $0.32$ in"
assert agrees(min(gaps), 0.03, 2) and agrees(max(gaps), 0.05, 2)  # book: appendix_f.tex "the two columns differ by only $0.03$ to $0.05$. That"
assert gaps[0] == max(gaps)                               # the gap is widest at -0.5
assert agrees(quad_var_r0(0.0), 1.39, 2) and agrees(v0, 1.30, 2) and agrees(quad_var_r0(0.0) - v0, 0.09, 2)  # book: appendix_f.tex "Carlo variance is the exact $1.39$, and its gap from $1.30$ is the $0.09$"
assert agrees(sd_delta[0.0], 1.14, 2) and agrees(sd_mc[-0.5], 0.87, 2)  # book: appendix_f.tex "independence there would report $1.14$ against the Monte Carlo $0.87$, about"
assert round(100 * (sd_delta[0.0] / sd_mc[-0.5] - 1)) == 31   # "about 31% too high"

# Self-check F.3(b): corr(beta, tau_R) = 0.3, relative form
RHO_SC_F = 0.3
assert RHO_SC_F == 0.3  # book: appendix_f.tex "suppose $\corr(\beta,\tau_R) = 0.3$. Use the"
cross = 2 * Fr(RHO_SC_F).limit_denominator(100) * cvb * cvt
assert RHO_SC_F == 0.3 and (cvb.denominator, cvt.numerator, cross.numerator, cross.denominator) == (3, 3, 3, 35)  # book: appendix_f.tex "$2 \times 0.3 \times (1/3)(3/7) = 3/35$ and"
cv2_sc = Fr(1, 9) + Fr(9, 49) + Fr(3, 35)
assert (cv2_sc.numerator, cv2_sc.denominator) == (839, 2205) and agrees(float(cv2_sc), 0.380, 3)  # book: appendix_f.tex "\approx 1/9 + 9/49 + 3/35 = 839/2205 \approx 0.380$, which gives"
assert agrees(np.sqrt(delta_cv2_r0(RHO_SC_F)), 0.617, 3) and agrees(R0_NOM, 2.1, 1)  # book: appendix_f.tex "$\CV_{\cR_0} \approx 0.617$. Multiplying by $\cR_0 = 2.1$ gives"
assert agrees(np.sqrt(delta_cv2_r0(RHO_SC_F)) * R0_NOM, 1.30, 2) and agrees(sd_delta[0.0], 1.14, 2)  # book: appendix_f.tex "$\sigma_{\cR_0} \approx 1.30$, up from $1.14$ for independent inputs: a"
print(f'Appendix F, correlated delta method: sigma(R0) delta {[round(sd_delta[r], 4) for r in (-0.5, 0, 0.5)]}, '
      f'MC {[round(sd_mc[r], 4) for r in (-0.5, 0, 0.5)]}; all checks pass')